# 03 - Retrieval Evaluation

Compare **text**, **vector**, and **hybrid** retrieval using hit-rate and MRR against the ground truth from notebook 02.

Run `make download` (ONNX model) and `make ingest` first.

In [ ]:
import sys
sys.path.append('../cocktail_assistant')

from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
import pandas as pd
from ingest import load_data, build_index, build_vector_index
from embedder import Embedder

documents = load_data('../data/cocktails.csv')
ground_truth = pd.read_csv('../data/ground_truth.csv').to_dict('records')
for r in ground_truth:
    r['id'] = str(r['id'])
len(documents), len(ground_truth)

## Build the three indexes

In [ ]:
text_index = build_index(documents)

embedder = Embedder('../models/Xenova/all-MiniLM-L6-v2')
vector_index = build_vector_index(documents, embedder)

## Retrieval functions
Reuse the boosts and RRF logic from `rag_helper`.

In [ ]:
BOOST = {"name": 3.0, "ingredients": 2.0, "category": 0.5}


def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results, boost_dict=BOOST)


def vector_search(query, num_results=5):
    return vector_index.search(embedder.encode(query), num_results=num_results)


def hybrid_search(query, num_results=5, k=60):
    text_results = text_index.search(query, num_results=10, boost_dict=BOOST)
    vector_results = vector_index.search(embedder.encode(query), num_results=10)

    scores, docs = {}, {}
    for results in (text_results, vector_results):
        for rank, doc in enumerate(results):
            doc_id = doc["id"]
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
            docs[doc_id] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[i] for i in ranked[:num_results]]

In [ ]:
def hit_rate(relevance_total):
    return sum(any(line) for line in relevance_total) / len(relevance_total)


def mrr(relevance_total):
    total = 0.0
    for line in relevance_total:
        for rank, hit in enumerate(line):
            if hit:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)


def evaluate(ground_truth, search_fn):
    relevance_total = []
    for row in ground_truth:
        results = search_fn(row["question"])
        relevance = [d["id"] == row["id"] for d in results]
        relevance_total.append(relevance)
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}

## Evaluate

In [ ]:
results = {
    "text": evaluate(ground_truth, text_search),
    "vector": evaluate(ground_truth, vector_search),
    "hybrid": evaluate(ground_truth, hybrid_search),
}
pd.DataFrame(results).T

The hybrid retriever combines lexical (TF-IDF) and semantic (ONNX embedding) signals with Reciprocal Rank Fusion and is used in the production assistant.